# YOLOv8 Fine-Tuning for MTSD Traffic Signs

This notebook converts the MTSD annotations into YOLO format, keeps only the high-level class name, and fine-tunes a YOLOv8 detector.

- `other-sign` stays as `other-sign`.
- Labels like `complementary--maximum-speed-limit-70--g1` become `complementary`.
- Panorama boxes with `cross_boundary` are split into two YOLO boxes.

## Imports and paths

The helper below finds the repo root whether the notebook starts from the workspace root or from the `code/` folder.

In [1]:
import json
import os
import re
import shutil
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO


PROJECT_ROOT = Path('/home/minh-le-vo-nhat/Documents/Minh-DUT/Ky-8-2025-2026/BigData/BTNhom')
DATA_ROOT = PROJECT_ROOT / 'Data'
CODE_ROOT = PROJECT_ROOT / 'code'

IMAGES_SPLIT_DIRS = {
    'train': DATA_ROOT / 'mtsd_fully_annotated_images.train.full' / 'images',
    'val': DATA_ROOT / 'mtsd_fully_annotated_images.val' / 'images',
    'test': DATA_ROOT / 'mtsd_fully_annotated_images.test' / 'images',
}

ANNOTATION_ROOT = DATA_ROOT / 'mtsd_fully_annotated_annotation' / 'mtsd_v2_fully_annotated'
ANNOTATIONS_DIR = ANNOTATION_ROOT / 'annotations'
SPLITS_DIR = ANNOTATION_ROOT / 'splits'
SUMMARY_CSV = CODE_ROOT / 'Data_summary.csv'

PREPARED_ROOT = CODE_ROOT / 'yolo_mtsd_highlevel'
YAML_PATH = PREPARED_ROOT / 'traffic_signs.yaml'
CLASS_MAP_PATH = PREPARED_ROOT / 'class_names.json'

HIGH_LEVEL_RE = re.compile(r'^([A-Za-z]+)--([A-Za-z0-9-]+)--g\d+$')



/home/minh-le-vo-nhat/anaconda3/envs/ts-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def normalize_label(raw_label: str) -> str:
    if raw_label == 'other-sign':
        return raw_label
    match = HIGH_LEVEL_RE.match(raw_label)
    if match:
        return match.group(1)
    return raw_label.split('--', 1)[0]


def split_keys(split_name: str) -> list[str]:
    split_file = SPLITS_DIR / f'{split_name}.txt'
    return [line.strip() for line in split_file.read_text().splitlines() if line.strip()]


def find_image_path(image_key: str) -> Path:
    for split_dir in IMAGES_SPLIT_DIRS.values():
        candidate = split_dir / f'{image_key}.jpg'
        if candidate.exists():
            return candidate
    for split_dir in IMAGES_SPLIT_DIRS.values():
        matches = list(split_dir.glob(f'{image_key}.*'))
        if matches:
            return matches[0]
    raise FileNotFoundError(f'Could not find image for key {image_key}')


def iter_bbox_parts(bbox: dict) -> list[dict]:
    if 'cross_boundary' in bbox:
        return [bbox['cross_boundary']['left'], bbox['cross_boundary']['right']]
    return [bbox]


def bbox_to_yolo(xmin: float, ymin: float, xmax: float, ymax: float, width: int, height: int):
    xmin = max(0.0, min(float(xmin), float(width)))
    xmax = max(0.0, min(float(xmax), float(width)))
    ymin = max(0.0, min(float(ymin), float(height)))
    ymax = max(0.0, min(float(ymax), float(height)))
    if xmax <= xmin or ymax <= ymin:
        return None
    x_center = ((xmin + xmax) / 2.0) / width
    y_center = ((ymin + ymax) / 2.0) / height
    box_width = (xmax - xmin) / width
    box_height = (ymax - ymin) / height
    return x_center, y_center, box_width, box_height

## Inspect the coarse class distribution

The CSV contains the original fine-grained labels. Here we collapse them to the high-level class names used for YOLO training.

In [5]:
summary_df = pd.read_csv(SUMMARY_CSV)
summary_df['coarse_label'] = summary_df['labels'].map(normalize_label)
coarse_summary = (
    summary_df.groupby('coarse_label', as_index=False)['counts']
    .sum()
    .sort_values('counts', ascending=False)
)
print(coarse_summary.to_string(index=False))

 coarse_label  counts
   other-sign  118782
   regulatory   31579
      warning   14333
complementary    9082
  information    6510


## Build the class map

This scans the annotation files, applies the label reduction rule, and creates a stable class order. `other-sign` is kept as a real class, not dropped.

In [6]:
annotation_files = sorted(ANNOTATIONS_DIR.glob('*.json'))
class_counter = Counter()

for ann_path in tqdm(annotation_files, desc='Scanning annotation classes'):
    ann_data = json.loads(ann_path.read_text())
    for obj in ann_data.get('objects', []):
        class_counter[normalize_label(obj['label'])] += 1

class_names = ['other-sign'] + sorted(name for name in class_counter if name != 'other-sign')
class_to_id = {name: idx for idx, name in enumerate(class_names)}

PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
CLASS_MAP_PATH.write_text(json.dumps(class_names, indent=2))

print(f'Classes: {len(class_names)}')
print(class_names)
print()
print('Top classes by object count:')
print(pd.Series(class_counter).reindex(class_names).fillna(0).astype(int).to_string())


Scanning annotation classes: 100%|██████████| 41909/41909 [00:02<00:00, 14181.65it/s]

Classes: 5
['other-sign', 'complementary', 'information', 'regulatory', 'warning']

Top classes by object count:
other-sign       135958
complementary     10405
information        7455
regulatory        36167
warning           16401


## Convert MTSD annotations to YOLO format

The notebook creates a new dataset layout under `code/yolo_mtsd_highlevel/` with symlinked images and YOLO label text files. If symlinks are not available, it falls back to copying the image files.

In [7]:
if PREPARED_ROOT.exists():
    shutil.rmtree(PREPARED_ROOT)

for split_name in ['train', 'val']:
    (PREPARED_ROOT / 'images' / split_name).mkdir(parents=True, exist_ok=True)
    (PREPARED_ROOT / 'labels' / split_name).mkdir(parents=True, exist_ok=True)

split_stats = {}

for split_name in ['train', 'val']:
    image_keys = split_keys(split_name)
    image_out_dir = PREPARED_ROOT / 'images' / split_name
    label_out_dir = PREPARED_ROOT / 'labels' / split_name
    class_stats = Counter()
    object_total = 0

    for image_key in tqdm(image_keys, desc=f'Preparing {split_name}'):
        image_path = find_image_path(image_key)
        annotation_path = ANNOTATIONS_DIR / f'{image_key}.json'
        if not annotation_path.exists():
            raise FileNotFoundError(f'Missing annotation file: {annotation_path}')

        output_image_path = image_out_dir / image_path.name
        if output_image_path.exists() or output_image_path.is_symlink():
            output_image_path.unlink()

        try:
            os.symlink(image_path.resolve(), output_image_path)
        except OSError:
            shutil.copy2(image_path, output_image_path)

        ann_data = json.loads(annotation_path.read_text())
        width = int(ann_data['width'])
        height = int(ann_data['height'])
        yolo_lines = []

        for obj in ann_data.get('objects', []):
            coarse_label = normalize_label(obj['label'])
            class_id = class_to_id[coarse_label]
            bbox = obj['bbox']

            for part in iter_bbox_parts(bbox):
                yolo_box = bbox_to_yolo(
                    part['xmin'],
                    part['ymin'],
                    part['xmax'],
                    part['ymax'],
                    width,
                    height,
                )
                if yolo_box is None:
                    continue
                x_center, y_center, box_width, box_height = yolo_box
                yolo_lines.append(
                    f'{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}'
                )
                class_stats[coarse_label] += 1
                object_total += 1

        label_path = label_out_dir / f'{image_key}.txt'
        label_path.write_text(('\n'.join(yolo_lines) + '\n') if yolo_lines else '')

    split_stats[split_name] = {'images': len(image_keys), 'objects': object_total, 'class_stats': class_stats}

for split_name, stats in split_stats.items():
    print(f"{split_name}: {stats['images']} images, {stats['objects']} YOLO boxes")
    print(pd.Series(stats['class_stats']).reindex(class_names).fillna(0).astype(int).to_string())
    print()

Preparing val: 100%|██████████| 5320/5320 [00:02<00:00, 1935.09it/s]

train: 36589 images, 180288 YOLO boxes
other-sign       118783
complementary      9082
information        6510
regulatory        31580
warning           14333

val: 5320 images, 26101 YOLO boxes
other-sign       17177
complementary     1323
information        945
regulatory        4588
warning           2068



## Write the YOLO data config and visualize one converted sample

This confirms the label conversion before training starts.

In [3]:
yaml_text = f'''path: {PREPARED_ROOT.as_posix()}
train: images/train
val: images/val
nc: {len(class_names)}
names: {class_names}
'''
YAML_PATH.write_text(yaml_text)
print(YAML_PATH)
print(yaml_text)

def draw_sample(split_name: str = 'train', index: int = 0):
    image_dir = PREPARED_ROOT / 'images' / split_name
    label_dir = PREPARED_ROOT / 'labels' / split_name
    image_files = sorted(image_dir.glob('*'))
    if not image_files:
        raise FileNotFoundError(f'No images found in {image_dir}')
    image_path = image_files[index]
    label_path = label_dir / f'{image_path.stem}.txt'

    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')

    if label_path.exists() and label_path.read_text().strip():
        for line in label_path.read_text().splitlines():
            class_id, x_center, y_center, box_width, box_height = map(float, line.split())
            class_id = int(class_id)
            h, w = image.shape[:2]
            xmin = int((x_center - box_width / 2) * w)
            ymin = int((y_center - box_height / 2) * h)
            xmax = int((x_center + box_width / 2) * w)
            ymax = int((y_center + box_height / 2) * h)
            cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
            cv2.putText(
                image,
                class_names[class_id],
                (xmin, max(0, ymin - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2,
                cv2.LINE_AA,
            )

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'{split_name}: {image_path.name}')
    plt.show()

draw_sample('train', 0)

NameError: name 'class_names' is not defined

## Fine-tune YOLOv8

This uses the lightweight `yolov8n.pt` checkpoint as a starting point. Increase the model size later if you want higher accuracy and have the GPU budget for it.

In [ ]:
device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 8 if device != 'cpu' else 4
workers = max(2, (os.cpu_count() or 4) // 2)

model = YOLO('yolov8n.pt')
model.train(
    data=str(YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=batch_size,
    device=device,
    workers=workers,
    project=str(CODE_ROOT / 'runs'),
    name='traffic_sign_yolov8_highlevel',
    exist_ok=True,
    patience=15,
)

Ultralytics 8.4.53 🚀 Python-3.14.4 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3770MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/minh-le-vo-nhat/Documents/Minh-DUT/Ky-8-2025-2026/BigData/BTNhom/code/yolo_mtsd_highlevel/traffic_signs.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam

## Validate and run inference

After training, the best weights are saved under `code/runs/traffic_sign_yolov8_highlevel/weights/best.pt`.

In [ ]:
best_weights = CODE_ROOT / 'runs' / 'traffic_sign_yolov8_highlevel' / 'weights' / 'best.pt'
if not best_weights.exists():
    raise FileNotFoundError(f'Missing trained weights: {best_weights}')

trained_model = YOLO(str(best_weights))
metrics = trained_model.val(data=str(YAML_PATH), split='val')
print(metrics)

test_images = sorted((PREPARED_ROOT / 'images' / 'val').glob('*'))
if test_images:
    sample_image = test_images[0]
    prediction = trained_model.predict(source=str(sample_image), conf=0.25, save=True)
    print(f'Predicted on {sample_image.name}')
    print(prediction)

Ultralytics 8.4.53 🚀 Python-3.14.4 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3770MiB)
Model summary (fused): 93 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 272.7±50.9 MB/s, size: 633.4 KB)
val: Scanning /home/minh-le-vo-nhat/Documents/Minh-DUT/Ky-8-2025-2026/BigData/BTNhom/code/yolo_mtsd_highlevel/labels/val.cache... 5320 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5320/5320 286.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 32% ━━━╸──────── 105/333 1.2s/it 1:25<4:43


OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 242.88 MiB is free. Process 12008 has 1.73 GiB memory in use. Including non-PyTorch memory, this process has 1.68 GiB memory in use. Of the allocated memory 1.47 GiB is allocated by PyTorch, and 130.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)